In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D10 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D10 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import hashlib
import json
import re
import unicodedata

import pandas as pd


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D10"
DOCUMENT_NAME = "IMPI — Inquérito Mensal à Produção Industrial"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11,
}

FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location",
]

MANDATORY_STRING_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Expected Value Type",
    "Source Location",
]

NULLABLE_STRING_FIELDS = ["Code"]

BASE_IDENTITY_FIELDS = [
    "Category",
    "Field or Concept",
]


DUPLICATE_DISAMBIGUATION_FIELD = "Section"

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Field or Concept",
    "Code",
    "Expected Value Type",
    "Source Location",
]

DIAGNOSTIC_FIELDS = [
    "Section",
    "Description",
]

OUTPUT_DIR = Path("outputs_D10_validation_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D10_reference_values.csv
#   2) D10_branch_B_parsed_extraction.json
#   3) D10_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    name for name in uploaded_files
    if name.lower().endswith(".csv")
]

json_files = [
    name for name in uploaded_files
    if name.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D10 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:

    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structurally_evaluable" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D10 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D10 Branch B technical diagnostics."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)


In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=True,
    encoding="utf-8-sig",
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    parsed_extraction = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)

for artefact_name, artefact in {
    "parsed extraction": parsed_extraction,
    "technical diagnostics": technical_diagnostics,
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

extracted_records = parsed_extraction.get("records")

if not isinstance(extracted_records, list):
    raise ValueError(
        "Parsed Branch B extraction must contain a records list."
    )

extracted_df = pd.DataFrame(extracted_records)

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256":
        sha256_file(PARSED_EXTRACTION_FILE),
    "technical_diagnostics_file": TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE),
}

print("Reference shape:", reference_df.shape)
print("Extraction shape:", extracted_df.shape)


In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status and verify inputs
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(structurally_evaluable)

schema_diagnostics = {
    "valid_json": bool(
        technical_diagnostics.get("valid_json", False)
    ),
    "record_schema_valid": bool(
        technical_diagnostics.get("record_schema_valid", False)
    ),
    "field_types_valid": bool(
        technical_diagnostics.get("field_types_valid", False)
    ),
    "structurally_evaluable": structurally_evaluable,
    "schema_validity": schema_validity,
}

if not structurally_evaluable:
    raise ValueError(
        "D10 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

if not reference_schema_exact:
    raise ValueError(
        "D10 Stage 1 reference schema does not match the fixed field list."
    )

if len(reference_df) != EXPECTED_RECORD_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_RECORD_COUNT} Stage 1 records; "
        f"found {len(reference_df)}."
    )

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
)

if reference_category_counts != EXPECTED_CATEGORY_COUNTS:
    raise ValueError(
        "D10 Stage 1 category counts do not match the fixed design."
    )

missing_extraction_fields = [
    field for field in FIELDS
    if field not in extracted_df.columns
]

extracted_comparison_df = extracted_df.copy(deep=True)

for field in missing_extraction_fields:
    extracted_comparison_df[field] = None

extracted_comparison_df = (
    extracted_comparison_df[FIELDS].copy()
)


extracted_df = extracted_comparison_df.copy(deep=True)

extraction_category_counts = (
    extracted_df["Category"]
    .value_counts(dropna=False)
    .to_dict()
)

content_diagnostics = {
    "reference_record_count_valid": True,
    "reference_category_counts_valid": True,
    "extraction_record_count_valid":
        len(extracted_df) == EXPECTED_RECORD_COUNT,
    "extraction_category_counts_valid":
        extraction_category_counts == EXPECTED_CATEGORY_COUNTS,
    "branch_B_scope_complete":
        technical_diagnostics.get("scope_complete"),
    "branch_B_content_diagnostics":
        technical_diagnostics.get("content_diagnostics"),
    "missing_extraction_columns":
        missing_extraction_fields,
}

print(json.dumps(
    schema_diagnostics,
    indent=2,
    ensure_ascii=False
))
print(json.dumps(
    content_diagnostics,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 5. Confirm fixed Stage 1 D10 reference semantics
# ============================================================

EXPECTED_UAE_TEMPLATE_LABELS = {
    "Código da UAE",
    "Designação da UAE",
    "Situação da UAE perante a atividade",
    "Observações da UAE",
    "Confirmar",
    "Produtos",
}

EXPECTED_PRODUCT_TABLE_LABELS = {
    "NIF",
    "UAE",
    "Período de Referência",
    "Nº",
    "Produto",
    "Unid.",
    "Código",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços",
    "Observações empresa",
    "Observações INE",
}


observed_uae_template_labels = set(
    reference_df.loc[
        reference_df["Category"]
        == "UAE template element",
        "Field or Concept",
    ]
)

observed_product_table_labels = set(
    reference_df.loc[
        reference_df["Category"]
        == "Product table field",
        "Field or Concept",
    ]
)


reference_period_field_valid = (
    (
        reference_df["Field or Concept"]
        == "Referência dos dados"
    ).sum()
    == 1
)

source_location_pattern_valid = bool(
    reference_df["Source Location"]
    .fillna("")
    .str.match(r"^PDF page [1-4] — .+$")
    .all()
)


reference_semantic_checks = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_schema_exact":
        bool(reference_schema_exact),

    "reference_types_valid":
        bool(reference_types_valid),

    "uae_template_valid":
        observed_uae_template_labels
        == EXPECTED_UAE_TEMPLATE_LABELS,

    "reference_period_field_valid":
        bool(reference_period_field_valid),

    "product_table_structure_valid":
        observed_product_table_labels
        == EXPECTED_PRODUCT_TABLE_LABELS,

    "source_location_pattern_valid":
        bool(source_location_pattern_valid),
}


reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "Corrected/current D10 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D10 reference does not match the current "
        "frozen Stage 1 D10 reference semantics."
    )


In [ ]:
# ============================================================
# 6. Comparison-only normalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = (
        text
        .replace("—", "-")
        .replace("–", "-")
        .replace("‑", "-")
        .replace("“", '"')
        .replace("”", '"')
        .replace("’", "'")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def normalise_identity_text(value):
    text = normalise_text(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


SECTION_EQUIVALENCE_GROUPS = [
    {
        "header",
        "questionnaire header",
        "legal notice",
    },
    {
        "response contacts",
        "contact and response information",
        "response information",
    },
    {
        "reference data",
        "reference-data area",
    },
    {
        "identification of statistical unit",
        "section i",
        "i - identificação da unidade estatística",
        "identificação da unidade estatística",
    },
    {
        "activity status",
        "section ii",
        "ii - situação da unidade estatística no período de referência dos dados",
        "situação da unidade estatística no período de referência dos dados",
    },
    {
        "observations",
        "section iii",
        "iii - observações",
        "observações",
    },
    {
        "responsible person",
        "section iv",
        "iv - responsável pelo preenchimento",
        "responsável pelo preenchimento",
    },
    {
        "uae information",
        "uae template",
    },
    {
        "product production table",
        "product table",
        "product table header",
        "product table columns",
    },
    {
        "filling instructions",
        "instruções de preenchimento",
    },
    {
        "explanatory notes",
        "notas explicativas",
    },
]


def canonical_section(value):
    value_norm = normalise_identity_text(value)

    for group_index, group in enumerate(
        SECTION_EQUIVALENCE_GROUPS
    ):
        normalised_group = {
            normalise_identity_text(item)
            for item in group
        }

        if value_norm in normalised_group:
            return f"section_group_{group_index}"

    return value_norm


def normalise_code(value):
    if value is None:
        return None

    text = normalise_text(value)

    if not text:
        return None

    return text.upper()


def extract_pdf_page(value):
    text = normalise_text(value)

    match = re.search(
        r"(?:physical\s+)?pdf\s+page\s+(\d+)",
        text,
    )

    if not match:
        return None

    return int(match.group(1))


def description_similarity(a, b):
    a_norm = normalise_text(a)
    b_norm = normalise_text(b)

    if not a_norm and not b_norm:
        return 1.0

    if not a_norm or not b_norm:
        return 0.0

    return SequenceMatcher(
        None,
        a_norm,
        b_norm,
    ).ratio()


In [ ]:
# ============================================================
# 7. Deterministic one-to-one identity alignment
# ============================================================


def base_identity(record):
    return (
        normalise_identity_text(
            record["Category"]
        ),
        normalise_identity_text(
            record["Field or Concept"]
        ),
    )


def section_identity(record):
    return canonical_section(
        record["Section"]
    )


reference_records = (
    reference_df
    .to_dict("records")
)

extraction_records = (
    extracted_df
    .to_dict("records")
)


reference_groups = defaultdict(list)
extraction_groups = defaultdict(list)

for index, record in enumerate(reference_records):
    reference_groups[
        base_identity(record)
    ].append(index)

for index, record in enumerate(extraction_records):
    extraction_groups[
        base_identity(record)
    ].append(index)


matches = []
matched_reference = set()
matched_extraction = set()
ambiguous_alignment_groups = []


all_base_keys = sorted(
    set(reference_groups)
    | set(extraction_groups)
)


for key in all_base_keys:

    ref_indices = reference_groups.get(
        key,
        [],
    )

    ext_indices = extraction_groups.get(
        key,
        [],
    )

    if (
        len(ref_indices) == 1
        and len(ext_indices) == 1
    ):
        r_idx = ref_indices[0]
        e_idx = ext_indices[0]

        matches.append({
            "Reference Index": r_idx,
            "Extraction Index": e_idx,
            "Alignment Rule":
                "Category + Field or Concept",
        })

        matched_reference.add(r_idx)
        matched_extraction.add(e_idx)
        continue


    ref_by_section = defaultdict(list)
    ext_by_section = defaultdict(list)

    for r_idx in ref_indices:
        ref_by_section[
            section_identity(
                reference_records[r_idx]
            )
        ].append(r_idx)

    for e_idx in ext_indices:
        ext_by_section[
            section_identity(
                extraction_records[e_idx]
            )
        ].append(e_idx)


    for section_key in sorted(
        set(ref_by_section)
        | set(ext_by_section)
    ):

        r_list = ref_by_section.get(
            section_key,
            [],
        )

        e_list = ext_by_section.get(
            section_key,
            [],
        )

        if (
            len(r_list) == 1
            and len(e_list) == 1
        ):
            r_idx = r_list[0]
            e_idx = e_list[0]

            matches.append({
                "Reference Index": r_idx,
                "Extraction Index": e_idx,
                "Alignment Rule":
                    "Category + Field or Concept + canonical Section",
            })

            matched_reference.add(r_idx)
            matched_extraction.add(e_idx)

        elif r_list or e_list:
            ambiguous_alignment_groups.append({
                "base_identity": key,
                "canonical_section": section_key,
                "reference_indices": r_list,
                "extraction_indices": e_list,
            })


missing_reference_indices = sorted(
    set(range(len(reference_records)))
    - matched_reference
)

unsupported_extraction_indices = sorted(
    set(range(len(extraction_records)))
    - matched_extraction
)


print("Aligned:", len(matches))
print("Missing:", len(missing_reference_indices))
print(
    "Unsupported/unmatched:",
    len(unsupported_extraction_indices),
)
print(
    "Ambiguous identity groups:",
    len(ambiguous_alignment_groups),
)


In [ ]:
# ============================================================
# 8. Field comparison
# ============================================================

def exact_text_equal(a, b):
    return (
        normalise_text(a)
        == normalise_text(b)
    )


def section_equal(a, b):
    return (
        canonical_section(a)
        == canonical_section(b)
    )


def code_equal(a, b):
    return (
        normalise_code(a)
        == normalise_code(b)
    )


def source_location_equal(a, b):

    page_a = extract_pdf_page(a)
    page_b = extract_pdf_page(b)

    return (
        page_a is not None
        and page_b is not None
        and page_a == page_b
    )


comparison_rows = []


for match in matches:

    r = reference_records[
        match["Reference Index"]
    ]

    e = extraction_records[
        match["Extraction Index"]
    ]

    out = {
        "Reference Index":
            match["Reference Index"],

        "Extraction Index":
            match["Extraction Index"],

        "Alignment Rule":
            match["Alignment Rule"],
    }


    correctness = {
        "Category":
            exact_text_equal(
                r["Category"],
                e["Category"],
            ),

        "Section":
            section_equal(
                r["Section"],
                e["Section"],
            ),

        "Field or Concept":
            exact_text_equal(
                r["Field or Concept"],
                e["Field or Concept"],
            ),

        "Description":
            exact_text_equal(
                r["Description"],
                e["Description"],
            ),

        "Code":
            code_equal(
                r["Code"],
                e["Code"],
            ),

        "Expected Value Type":
            exact_text_equal(
                r["Expected Value Type"],
                e["Expected Value Type"],
            ),

        "Source Location":
            source_location_equal(
                r["Source Location"],
                e["Source Location"],
            ),
    }


    for field in FIELDS:
        out[
            f"Reference {field}"
        ] = r[field]

        out[
            f"Extracted {field}"
        ] = e[field]

        out[
            f"{field} Correct"
        ] = bool(
            correctness[field]
        )


    out[
        "Description Similarity Diagnostic"
    ] = float(
        description_similarity(
            r["Description"],
            e["Description"],
        )
    )


    out[
        "Fully Correct Primary Record"
    ] = all(
        correctness[field]
        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )


    comparison_rows.append(out)


comparison_df = pd.DataFrame(
    comparison_rows
)


missing_records_df = (
    reference_df
    .iloc[
        missing_reference_indices
    ]
    .copy()
)


unsupported_records_df = (
    extracted_df
    .iloc[
        unsupported_extraction_indices
    ]
    .copy()
)


discrepant_records_df = (
    comparison_df.loc[
        ~comparison_df[
            "Fully Correct Primary Record"
        ]
    ]
    .copy()
)


print(
    "Fully correct primary records:",
    int(
        comparison_df[
            "Fully Correct Primary Record"
        ].sum()
    )
)

print(
    "Primary discrepant records:",
    len(discrepant_records_df),
)


In [ ]:
# ============================================================
# 9. Calculate common validation metrics
# ============================================================

aligned_records = len(comparison_df)

fully_correct_records = int(
    comparison_df[
        "Fully Correct Primary Record"
    ].sum()
)

discrepant_records = (
    aligned_records
    - fully_correct_records
)

missing_records = len(
    missing_reference_indices
)

unsupported_records = len(
    unsupported_extraction_indices
)

N_REF = int(len(reference_df))
N_EXT = int(len(extracted_df))

completeness = (
    aligned_records / N_REF
    if N_REF
    else 0.0
)

missing_rate = (
    missing_records / N_REF
    if N_REF
    else 0.0
)

record_precision_exact = (
    fully_correct_records / N_EXT
    if N_EXT
    else 0.0
)

record_recall_exact = (
    fully_correct_records / N_REF
    if N_REF
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)

unsupported_rate = (
    unsupported_records / N_EXT
    if N_EXT
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_records / aligned_records
    if aligned_records
    else 0.0
)

primary_field_accuracy_among_aligned = {
    field: (
        float(
            comparison_df[
                f"{field} Correct"
            ].mean()
        )
        if aligned_records
        else 0.0
    )
    for field in PRIMARY_CORRECTNESS_FIELDS
}

diagnostic_field_accuracy = {
    "Section": (
        float(
            comparison_df[
                "Section Correct"
            ].mean()
        )
        if aligned_records
        else 0.0
    ),
    "Description exact": (
        float(
            comparison_df[
                "Description Correct"
            ].mean()
        )
        if aligned_records
        else 0.0
    ),
    "Description mean lexical similarity": (
        float(
            comparison_df[
                "Description Similarity Diagnostic"
            ].mean()
        )
        if aligned_records
        else 0.0
    ),
}

correct_primary_field_instances = int(
    sum(
        comparison_df[
            f"{field} Correct"
        ].sum()
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
)

expected_primary_field_instances = int(
    N_REF * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_primary_field_instances
    / expected_primary_field_instances
    if expected_primary_field_instances
    else 0.0
)

category_metrics = {}

for category, expected in EXPECTED_CATEGORY_COUNTS.items():

    ref_subset = reference_df[
        reference_df["Category"] == category
    ]

    ext_subset = extracted_df[
        extracted_df["Category"] == category
    ]

    aligned_subset = comparison_df[
        comparison_df["Reference Category"] == category
    ]

    full = int(
        aligned_subset[
            "Fully Correct Primary Record"
        ].sum()
    )

    aligned_count = len(aligned_subset)

    p = (
        full / len(ext_subset)
        if len(ext_subset)
        else 0.0
    )

    r = (
        full / len(ref_subset)
        if len(ref_subset)
        else 0.0
    )

    category_metrics[category] = {
        "expected_records": int(expected),
        "extracted_records": int(len(ext_subset)),
        "aligned_records": int(aligned_count),
        "fully_correct_records": full,
        "discrepant_records":
            int(aligned_count - full),
        "completeness": (
            aligned_count / expected
            if expected
            else 0.0
        ),
        "record_precision_exact": p,
        "record_recall_exact": r,
        "record_f1_exact": (
            2 * p * r / (p + r)
            if p + r
            else 0.0
        ),
    }

print("Reference records:", N_REF)
print("Extracted records:", N_EXT)
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print("Field accuracy:", round(field_accuracy, 4))
print("Schema valid:", schema_validity)


In [ ]:
# ============================================================
# 10. Build final Branch B validation summary
# ============================================================

field_validation_df = pd.DataFrame([
    {
        "Field": field,
        "Role": (
            "Primary correctness"
            if field in PRIMARY_CORRECTNESS_FIELDS
            else "Diagnostic"
        ),
        "Correct": int(
            comparison_df[
                f"{field} Correct"
            ].sum()
        ),
        "Aligned Records": int(aligned_records),
        "Accuracy Among Aligned": (
            float(
                comparison_df[
                    f"{field} Correct"
                ].mean()
            )
            if aligned_records
            else 0.0
        ),
        "Overall Accuracy Against Reference": (
            float(
                comparison_df[
                    f"{field} Correct"
                ].sum()
                / N_REF
            )
            if N_REF
            else 0.0
        ),
    }
    for field in FIELDS
])

category_metrics_df = pd.DataFrame([
    {
        "Category": category,
        **values,
    }
    for category, values in category_metrics.items()
])

fully_correct_records_df = (
    comparison_df.loc[
        comparison_df[
            "Fully Correct Primary Record"
        ]
    ].copy()
)

summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": int(aligned_records),
    "fully_correct_records":
        int(fully_correct_records),
    "discrepant_records":
        int(discrepant_records),
    "missing_records":
        int(missing_records),
    "unsupported_extracted_records":
        int(unsupported_records),

    "completeness":
        round(completeness, 4),
    "missing_rate":
        round(missing_rate, 4),
    "record_precision_exact":
        round(record_precision_exact, 4),
    "record_recall_exact":
        round(record_recall_exact, 4),
    "record_f1_exact":
        round(record_f1_exact, 4),
    "unsupported_rate":
        round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned":
        round(discrepancy_rate_among_aligned, 4),
    "field_accuracy":
        round(field_accuracy, 4),

    "primary_field_accuracy_among_aligned": {
        field: round(value, 4)
        for field, value
        in primary_field_accuracy_among_aligned.items()
    },

    "diagnostic_field_accuracy": {
        field: round(value, 4)
        for field, value
        in diagnostic_field_accuracy.items()
    },

    "schema_validity":
        schema_validity,
    "schema_diagnostics":
        schema_diagnostics,
    "structurally_evaluable":
        structurally_evaluable,
    "content_diagnostics":
        content_diagnostics,

    "base_identity_fields":
        BASE_IDENTITY_FIELDS,
    "duplicate_disambiguation_field":
        DUPLICATE_DISAMBIGUATION_FIELD,
    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,
    "diagnostic_fields":
        DIAGNOSTIC_FIELDS,

    "matching_rules": {
        "base_identity":
            BASE_IDENTITY_FIELDS,
        "duplicate_disambiguation_field":
            DUPLICATE_DISAMBIGUATION_FIELD,
        "matching_method":
            (
                "Deterministic one-to-one identity alignment "
                "frozen from D10 Validation A"
            ),
        "value_fields_used_for_alignment":
            False,
    },

    "comparison_rules_frozen_from_branch_A":
        True,

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(reference_semantics_valid),
        "checks":
            reference_semantic_checks,
        "reference_modified_by_validation":
            False,
    },

    "category_metrics":
        category_metrics,

    "normalisation_note":
        (
            "Controlled deterministic normalisation was applied "
            "only for comparison and alignment. The preserved "
            "Branch B extraction was not manually corrected."
        ),

    "input_provenance":
        input_provenance,
}

print(json.dumps(
    summary,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 11. Validation integrity checks
# ============================================================

assert reference_semantics_valid
assert schema_validity

assert (
    aligned_records
    + missing_records
    == N_REF
)

assert (
    aligned_records
    + unsupported_records
    == N_EXT
)

assert (
    fully_correct_records
    + discrepant_records
    == aligned_records
)

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision_exact":
        record_precision_exact,
    "record_recall_exact":
        record_recall_exact,
    "record_f1_exact":
        record_f1_exact,
    "unsupported_rate":
        unsupported_rate,
    "discrepancy_rate":
        discrepancy_rate_among_aligned,
    "field_accuracy":
        field_accuracy,
}.items():

    assert (
        0.0 <= metric_value <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )

print("Validation integrity checks passed.")


In [ ]:
# ============================================================
# 12. Export validation artefacts
# ============================================================

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_validation_summary.json"
)

DETAILED_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_unsupported_records.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_field_validation.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_category_metrics.csv"
)

REFERENCE_SEMANTICS_PATH = (
    OUTPUT_DIR
    / "D10_reference_semantics_confirmation.json"
)

ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR
    / "D10_branch_B_alignment_issues.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

REFERENCE_SEMANTICS_PATH.write_text(
    json.dumps(
        {
            "document_id":
                DOCUMENT_ID,
            "reference_semantics_valid":
                reference_semantics_valid,
            "checks":
                reference_semantic_checks,
        },
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

ALIGNMENT_ISSUES_PATH.write_text(
    json.dumps(
        ambiguous_alignment_groups,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

generated_outputs = [
    SUMMARY_PATH,
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_VALIDATION_PATH,
    CATEGORY_METRICS_PATH,
    REFERENCE_SEMANTICS_PATH,
    ALIGNMENT_ISSUES_PATH,
]

print("D10 Validation B artefacts saved.")

for path in generated_outputs:
    print("-", path.name)


In [ ]:
# ============================================================
# 13. Download generated validation artefacts
# ============================================================

for path in generated_outputs:

    if path.exists():
        files.download(path)
